# score_runs_positive_control — GPT-4.1 OmniContext scoring of the memorization control

Same structure as `score_runs_lora_val`: one `STEPS` list, each checkpoint's directory scored in
turn, one score file per checkpoint, resumable per row. Same `evaluate()` break patch, same resize
convention, same ThreadPool scorer with the uid guard and the guessed-score / unparsable-score
filters as `score_runs`.

`DATASET=screen`, `RUN=positive_control_r8`, `BASE_RUN=baseline`. The 302 rows are the rows the
adapter was trained on, so the direction of the move is what the control tests: every arm landed at
or below the unadapted baseline, which leaves "these methods do not help in this domain" and "this
setup would not show an improvement even if one existed" indistinguishable.

Section 6 prints the checkpoint curve, the paired delta of each checkpoint against
`screen__baseline.jsonl` on the same rows, and a verdict over the set.

The comparison has a measured noise floor on this exact split: `screen__baseline__rerun`
regenerated and rescored these same 302 rows under an identical configuration and moved +0.055
with a row-level sigma of 0.966. That is what "no change" looks like here.

302 rows scored per checkpoint.

## 1. Repo, scorer patch

In [ ]:
import os, subprocess, sys
REPO = "/content/OmniGen2"
if not os.path.isdir(REPO):
    subprocess.run(["git","clone","https://github.com/VectorSpaceLab/OmniGen2.git",REPO], check=True)
subprocess.run([sys.executable,"-m","pip","install","-q","regex"], check=True)   # json_util imports `regex`
sys.path.insert(0, REPO)
os.chdir(REPO)

# without the break, evaluate() re-runs a successful parse to max_tries: 6 calls per row instead of 2
p = "/content/OmniGen2/omnicontext/omnicontext_score.py"
lines = open(p).read().split("\n")
if not any("added: stop after successful parse" in l for l in lines):
    out = []
    for ln in lines:
        out.append(ln)
        if 'raise Exception("rate_limit_exceeded")' in ln:
            out.append("                break   # added: stop after successful parse")
    open(p, "w").write("\n".join(out))
    print("patched")
else:
    print("already patched")

patched


## 2. Drive, run config

In [ ]:
from google.colab import drive, userdata
import os, json
drive.mount("/content/drive")
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

DRIVE    = "/content/drive/MyDrive/Colab Notebooks/My Master's Thesis/Data"
DATASET  = "screen"                  # the 302 rows the control was trained on
STEPS    = [100, 500]                # 5.3 and 26.5 epochs; add 200/300/400 if the two ends disagree
RUN      = "positive_control_r8"
BASE_RUN = "baseline"                # the unadapted model on the same 302 rows, already scored

for step in STEPS:
    gen = f"{DRIVE}/gen/{DATASET}/{RUN}_step{step}"
    n = len([f for f in os.listdir(gen) if f.endswith(".png")]) if os.path.isdir(gen) else 0
    man = f"{gen}/manifest.json"
    m = json.load(open(man)) if os.path.exists(man) else {}
    assert n, f"no generated images at {gen}; run positive_control_gen first"
    if m:
        assert m.get("step") == step, f"manifest step {m.get('step')} disagrees with {step}"
    print(f"step {step:>4d}: {n:>4d} images | {m.get('gpu','manifest absent')} | written {m.get('written','-')}")

base_scores = f"{DRIVE}/scores/{DATASET}__{BASE_RUN}.jsonl"
assert os.path.exists(base_scores), f"comparator missing: {base_scores}"
print(f"\ncomparator: {base_scores}")

Mounted at /content/drive
step  100:  302 images | NVIDIA A100-SXM4-40GB | written 2026-09-12 14:13:48
step  500:  302 images | NVIDIA A100-SXM4-40GB | written 2026-09-12 14:13:48

comparator: /content/drive/MyDrive/Colab Notebooks/My Master's Thesis/Data/scores/screen__baseline.jsonl


## 3. Reference images

In [ ]:
IMGS, EXPECTED = "/content/imgs", 37063
TAR = f"{DRIVE}/imgs_subset.tar"

def n_files(d): return sum(len(f) for _, _, f in os.walk(d)) if os.path.isdir(d) else 0

if n_files(IMGS) < EXPECTED:
    os.makedirs(IMGS, exist_ok=True)
    r = subprocess.run(f'cp "{TAR}" /content/imgs_subset.tar && tar -xf /content/imgs_subset.tar -C {IMGS}',
                       shell=True, capture_output=True, text=True)
    if r.returncode: raise RuntimeError(f"rc={r.returncode}\n{r.stderr[:800]}")
    print("restored", n_files(IMGS), "/", EXPECTED)
    assert n_files(IMGS) >= EXPECTED, "restore incomplete"
else:
    print("images complete:", n_files(IMGS))

for s in ["train", "val", "test", "screen", "screen_small"]:
    src = f"{DRIVE}/{s}.jsonl"
    if os.path.exists(src) and not os.path.exists(f"/content/{s}.jsonl"):
        subprocess.run(["cp", src, f"/content/{s}.jsonl"])

rows = [json.loads(l) for l in open(f"/content/{DATASET}.jsonl", encoding="utf-8")]
# indices generated across all step dirs (each step scored separately in the loop below)
all_have = sorted({int(f[:-4])
                   for step in STEPS
                   for f in os.listdir(f"{DRIVE}/gen/{DATASET}/{RUN}_step{step}")
                   if f.endswith(".png") and f[:-4].isdigit()})
assert all_have, f"no generated images found for {RUN} across steps {STEPS}"
assert max(all_have) < len(rows), f"image index {max(all_have)} outside {DATASET} ({len(rows)} rows)"
missing = [p for i in all_have for p in rows[i]["input_images"] if not os.path.exists(os.path.join(IMGS, p))]
assert not missing, f"{len(missing)} reference images missing, first: {missing[:3]}"
print(f"{DATASET}: {len(rows)} rows | {len(all_have)} generated indices across {len(STEPS)} steps")

restored 37063 / 37063
screen: 302 rows | 302 generated indices across 2 steps


## 4. Sanity — 3 rows

In [ ]:
import math
from PIL import Image
from omnicontext.omnicontext_score import OmniContextScore

scorer = OmniContextScore("https://api.openai.com/v1/chat/completions", userdata.get("OPENAI_API_KEY"))

def resize_ref(im):   # OmniContext default: cap at 1024^2, round to /16 — references only
    w, h = im.size
    cap = 1024*1024
    if w*h > cap:
        r = math.sqrt(cap/(w*h)); w, h = int(w*r), int(h*r)
    return im.resize((max(16, w//16*16), max(16, h//16*16)))

# 3-row smoke on the first step's images to confirm the scorer + adapter path before the full loop
_smoke_step = STEPS[0]
_smoke_GEN  = f"{DRIVE}/gen/{DATASET}/{RUN}_step{_smoke_step}"
_smoke_have = sorted(int(f[:-4]) for f in os.listdir(_smoke_GEN) if f.endswith(".png") and f[:-4].isdigit())
for i in _smoke_have[:3]:
    r = rows[i]
    refs = [resize_ref(Image.open(os.path.join(IMGS, p)).convert("RGB")) for p in r["input_images"]]
    gen  = Image.open(f"{_smoke_GEN}/{i:05d}.png").convert("RGB")
    res  = scorer.evaluate(refs + [gen], r["instruction"], with_scene=False)
    print(f"step {_smoke_step} row {i}: PF={res['PF_scores']['score']}  SC={res['SC_scores']['score']}")
    print(f"   PF: {str(res['PF_scores']['reasoning'])[:140]}")
    print(f"   SC: {str(res['SC_scores']['reasoning'])[:140]}\n")

step 100 row 0: PF=8  SC=3
   PF: The image accurately depicts a businesswoman in a modern office chair, wearing a tailored blazer, focused intently on her laptop with a pen 
   SC: The notepad in the final image is not a strong match for the reference notepad. The AI-generated scene shows a plain notepad, but the specif

step 100 row 1: PF=8  SC=8
   PF: The image successfully depicts a focused businesswoman in a modern office chair, dressed in a tailored blazer, and displaying a poised and p
   SC: The laptop in the final image closely resembles the sleek laptop in the reference image, matching in color, overall shape, and keyboard plac

step 100 row 2: PF=9  SC=9
   PF: The image depicts a businesswoman dressed in a tailored blazer, sitting in a modern office chair at a desk. She holds a pen over a notepad a
   SC: The businesswoman in the final image strongly resembles the woman in the reference image: facial structure, hair style and color, skin tone,



## 5. Full pass per checkpoint (resumable)

In [ ]:
import time
from concurrent.futures import ThreadPoolExecutor, as_completed

def num(x):           # fix_json can repair a bare score into [8]
    if isinstance(x, list) and x: x = x[0]
    return x if isinstance(x, (int, float)) else None

# identical scorer to score_runs.ipynb; only wrapped to run each step's directory in turn.
# rate limits crash a pass non-destructively: rows are flushed per row, the done-set skips them,
# error rows stay retryable -> just re-run this cell until every step prints "to score: 0".
for step in STEPS:
    GEN    = f"{DRIVE}/gen/{DATASET}/{RUN}_step{step}"
    SCORES = f"{DRIVE}/scores/{DATASET}__{RUN}_step{step}.jsonl"
    have   = sorted(int(f[:-4]) for f in os.listdir(GEN) if f.endswith(".png") and f[:-4].isdigit())

    done = set()
    if os.path.exists(SCORES):
        for l in open(SCORES, encoding="utf-8"):
            try:
                d = json.loads(l)
                if "error" not in d: done.add(d["i"])   # failures stay retryable
            except: pass

    def score_row(i):
        r  = rows[i]
        gp = f"{GEN}/{i:05d}.png"
        if not os.path.exists(gp): return {"i": i, "error": "missing_generated"}
        refs = [resize_ref(Image.open(os.path.join(IMGS, p)).convert("RGB")) for p in r["input_images"]]
        gen  = Image.open(gp).convert("RGB")            # official: generated image not resized
        time.sleep(0.2)
        res = scorer.evaluate(refs + [gen], r["instruction"], with_scene=False)   # original instruction, never the augmented one
        pf, sc = num(res["PF_scores"]["score"]), num(res["SC_scores"]["score"])
        if pf is None or sc is None: return {"i": i, "error": "unparsable_score"}
        # evaluate() substitutes random.randint(0,10) after 3 failed parses; that is not a measurement
        if any(str(res[k]["reasoning"]).startswith("guess_if_cannot_parse") for k in ("PF_scores", "SC_scores")):
            return {"i": i, "error": "guessed_score"}
        ups = ["_up" in os.path.basename(p) for p in r["input_images"]]
        return {"i": i, "uid": r["output_image"] + "|" + "|".join(sorted(r["input_images"])), "n_refs": len(r["input_images"]),
                "ref_kind": "all_up" if all(ups) else ("no_up" if not any(ups) else "mixed"),
                "PF": pf, "SC": sc,
                "PF_reason": res["PF_scores"]["reasoning"], "SC_reason": res["SC_scores"]["reasoning"]}

    todo = [i for i in have if i not in done]
    print(f"\nstep {step}: to score {len(todo)} (done: {len(done)}) -> {SCORES}")
    with open(SCORES, "a", encoding="utf-8") as f, ThreadPoolExecutor(max_workers=8) as ex:
        futs = [ex.submit(score_row, i) for i in todo]
        for n, fut in enumerate(as_completed(futs), 1):
            f.write(json.dumps(fut.result(), ensure_ascii=False) + "\n"); f.flush()
            if n % 25 == 0: print(f"  {n}/{len(todo)}")
    print(f"step {step} pass done -> {SCORES}")
print("\nall steps: re-run this cell until each prints 'to score 0'")


step 100: to score 302 (done: 0) -> /content/drive/MyDrive/Colab Notebooks/My Master's Thesis/Data/scores/screen__positive_control_r8_step100.jsonl
  25/302
  50/302
  75/302
  100/302
  125/302
  150/302
  175/302
  200/302
  225/302
  250/302
  275/302
  300/302
step 100 pass done -> /content/drive/MyDrive/Colab Notebooks/My Master's Thesis/Data/scores/screen__positive_control_r8_step100.jsonl

step 500: to score 302 (done: 0) -> /content/drive/MyDrive/Colab Notebooks/My Master's Thesis/Data/scores/screen__positive_control_r8_step500.jsonl
  25/302
  50/302
  75/302
  100/302
  125/302
  150/302
  175/302
  200/302
  225/302
  250/302
  275/302
  300/302
step 500 pass done -> /content/drive/MyDrive/Colab Notebooks/My Master's Thesis/Data/scores/screen__positive_control_r8_step500.jsonl

all steps: re-run this cell until each prints 'to score 0'


## 6. Checkpoint curve, paired delta vs BASE_RUN, verdict

In [ ]:
import statistics as st
from collections import defaultdict
from scipy.stats import ttest_rel

SIGMA_ITEM = 0.362      # judge re-scoring SD on an identical image; RMS over sd_reps (90 rows x5, ddof=1)
RERUN_SD   = 0.966      # row-level sigma of the identical-config rerun on these same 302 screen rows
RERUN_MEAN = 0.055      # and its mean shift, which is what "no change" looks like on this split

ov = lambda r: math.sqrt(max(r["PF"], 0) * max(r["SC"], 0))

def load(path):
    d, errs = {}, defaultdict(int)
    for l in open(path, encoding="utf-8"):
        r = json.loads(l)
        if "error" in r: errs[r["error"]] += 1; continue
        d[r["i"]] = r
    return d, errs

base, base_errs = load(f"{DRIVE}/scores/{DATASET}__{BASE_RUN}.jsonl")
print(f"{DATASET}/{BASE_RUN}: {len(base)} scored"
      + (f" | errors {dict(base_errs)}" if base_errs else "")
      + f" | PF {st.mean([d['PF'] for d in base.values()]):.3f}"
      + f"  SC {st.mean([d['SC'] for d in base.values()]):.3f}"
      + f"  Overall {st.mean([ov(d) for d in base.values()]):.3f}")

print(f"\n{'step':>6s} {'n':>4s} {'PF':>7s} {'SC':>7s} {'Overall':>8s} "
      f"{'dPF':>8s} {'dSC':>8s} {'dOverall':>9s} {'SE':>7s} {'p':>9s}")
curve = {}
for step in STEPS:
    cur, errs = load(f"{DRIVE}/scores/{DATASET}__{RUN}_step{step}.jsonl")
    I = sorted(set(cur) & set(base))
    # a row index means nothing on its own; the uid is what identifies the data row
    bad = [i for i in I if cur[i]["uid"] != base[i]["uid"]]
    assert not bad, f"step {step}: uid mismatch on rows {bad[:5]} — runs are not aligned"
    d_ov = [ov(cur[i]) - ov(base[i]) for i in I]
    se = st.stdev(d_ov) / math.sqrt(len(d_ov))
    p = ttest_rel([ov(cur[i]) for i in I], [ov(base[i]) for i in I])[1]
    curve[step] = dict(n=len(I), mean=st.mean(d_ov), sd=st.stdev(d_ov), se=se, p=p, cur=cur, I=I, errs=errs)
    print(f"{step:>6d} {len(I):>4d} "
          f"{st.mean([cur[i]['PF'] for i in I]):>7.3f} {st.mean([cur[i]['SC'] for i in I]):>7.3f} "
          f"{st.mean([ov(cur[i]) for i in I]):>8.3f} "
          f"{st.mean([cur[i]['PF']-base[i]['PF'] for i in I]):>+8.3f} "
          f"{st.mean([cur[i]['SC']-base[i]['SC'] for i in I]):>+8.3f} "
          f"{st.mean(d_ov):>+9.3f} {se:>7.3f} {p:>9.3g}")
    if errs: print(f"       errors {dict(errs)}")

print("\nby stratum (Overall delta):")
print(f"{'step':>6s} {'stratum':>10s} {'n':>4s} {'delta':>9s}")
for step in STEPS:
    cur, I = curve[step]["cur"], curve[step]["I"]
    for key in ["n_refs", "ref_kind"]:
        g = defaultdict(list)
        for i in I: g[cur[i][key]].append(ov(cur[i]) - ov(base[i]))
        for k, v in sorted(g.items(), key=str):
            print(f"{step:>6d} {str(k):>10s} {len(v):>4d} {st.mean(v):>+9.3f}")

best = max(STEPS, key=lambda s: curve[s]["mean"])
b = curve[best]
print(f"\n  sigma_d (Overall) at step {best} = {b['sd']:.3f}")
print(f"  identical-config rerun        = {RERUN_MEAN:+.3f} at sigma_d {RERUN_SD:.3f}, measured on these same rows")
print(f"  judge-only floor              = {math.sqrt(2)*SIGMA_ITEM:.3f}   (sqrt(2) * {SIGMA_ITEM})")
print(f"  detection threshold at step {best} = {1.96*b['se']:.3f}   (1.96 * SE)")

print("\nverdict:")
if b["mean"] > 0 and b["p"] < 0.05:
    print(f"  PASS. Step {best} raises Overall by {b['mean']:+.3f} (p {b['p']:.3g}) on rows the adapter")
    print(f"  was trained on, against {RERUN_MEAN:+.3f} for rerunning the baseline unchanged.")
    print("  The training-to-metric chain can move the measure upward, so the arms' null results")
    print("  are a property of those methods on this domain and not of the setup.")
elif b["mean"] > 0:
    print(f"  INCONCLUSIVE. Best is step {best} at {b['mean']:+.3f}, p {b['p']:.3g} — inside the noise")
    print("  on this split. Score an intermediate checkpoint before concluding either way.")
else:
    fall = [s for s in STEPS if curve[s]["mean"] < 0]
    print(f"  NO UPWARD MOVE at {'any scored checkpoint' if len(fall) == len(STEPS) else 'step ' + str(best)}.")
    print("  Read the curve before concluding. At step 500 the adapter sits at 656% of the arm's")
    print("  step-4000 norm, so a fall there alone is consistent with training past the peak rather")
    print("  than with an insensitive pipeline. A fall at the earliest checkpoint too is the real")
    print("  negative, and would need reporting as a limit on what this evaluation can detect.")

screen/baseline: 302 scored | PF 8.017  SC 8.079  Overall 7.955

  step    n      PF      SC  Overall      dPF      dSC  dOverall      SE         p
   100  302   7.954   7.444    7.568   -0.063   -0.636    -0.387   0.103  0.000215
   500  302   7.791   7.258    7.381   -0.225   -0.821    -0.574   0.107  1.42e-07

by stratum (Overall delta):
  step    stratum    n     delta
   100          1  152    -0.469
   100          2  120    -0.335
   100          3   30    -0.176
   100     all_up  119    -0.541
   100      mixed   73    -0.383
   100      no_up  110    -0.223
   500          1  152    -0.656
   500          2  120    -0.466
   500          3   30    -0.591
   500     all_up  119    -0.862
   500      mixed   73    -0.319
   500      no_up  110    -0.432

  sigma_d (Overall) at step 100 = 1.795
  identical-config rerun        = +0.055 at sigma_d 0.966, measured on these same rows
  judge-only floor              = 0.512   (sqrt(2) * 0.362)
  detection threshold at step 100 = 0.20